In [30]:
import torch.nn as nn
import torchvision.transforms.functional as TF

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_chanenels, out_channels):
        self.conv_double = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=0),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=0),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.conv2(x)
        x = self.conv2(x)
        
        return x

In [32]:
class DownSample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        fm = self.conv(x)
        p = self.pool(fm)
        return fm, p

In [33]:
class UpSample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # up, convTranspose, conv2d, conv2d
        self.up = nn.ConvTranspose2d(in_channels, in_channels//2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        x2 = TF.center_crop(x2, [x1.size()[2], x1.size()[3]])
        x = nn.cat([x1, x2], dim=1)
        return self.conv(x)

In [34]:
class UNet(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.down_conv_1 = DownSample(in_channels, 64)
        self.down_conv_2 = DownSample(64, 128)
        self.down_conv_3 = DownSample(128, 256)
        self.down_conv_4 = DownSample(256, 512)

        # 가장 가운데 있는 친구는 오로지 Conv-Conv
        self.bottle_neck = DoubleConv(512, 1024)

        self.up_conv_1 = UpSample(1024, 512)
        self.up_conv_2 = UpSample(512, 256)
        self.up_conv_3 = UpSample(256, 128)
        self.up_conv_4 = UpSample(128, 64)

        self.out = nn.Conv2d(in_channels=64, out_channels=num_classes, kernel_size=1)
    
    def forward (self, x):
        # 다운 4개, 중간 1개, 업 4개, 최종 출력 1개
        
        # 일단 DownSample했을 때 fm, p 2개가 나옴.
        # DownSample의 결과는 각 p이다.
        fm1, p1 = self.down_conv_1(x)
        fm2, p2 = self.down_conv_2(p1)
        fm3, p3 = self.down_conv_3(p2)
        fm4, p4 = self.down_conv_4(p3)
        
        b = self.bottle_neck(p4)

        up_1 = self.up_conv1(b, fm4)
        up_2 = self.up_conv_2(up_1, fm3)
        up_3 = self.up_conv_2(up_2, fm2)
        up_4 = self.up_conv_2(up_3, fm1)

        # 확대는 다 했으니, 최종 출력만
        out = self.out( up_4 )

        return out


In [35]:
from torchinfo import summary



In [36]:
model = UNet(in_channels=1, num_classes=2)
summary(model, input_size=(1, 1, 572, 572))

AttributeError: cannot assign module before Module.__init__() call